<a href="https://colab.research.google.com/github/funny1vamp/project-for-dl/blob/main/permuted_mnist_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1 Minute Permuted MNIST — 最终版本（线上 0.9876）

**模型配置**
- MLP：`784 → 256 → 256 → 10`，每个隐藏层 `Linear → BatchNorm1d → ReLU`
- 优化器：SGD + Nesterov 动量 0.9，峰值学习率 0.2
- 学习率：按**已用时间比例**做余弦退火（0.2 → 1e-5），训练预算 40 s
- 损失：交叉熵 + 标签平滑 0.1
- batch 256，使用全部 60000 个训练样本

**成绩**：本地 3 种子平均 0.9867（单次 0.9859–0.9873），线上 0.9876；基线逻辑回归为 0.922。

**必须保留的规则**：每次 `train()` 都新建模型；标准化统计量从传入数据现算；墙钟时间保护；`predict` 中使用 `eval()` 和 `no_grad()`。

## 1. 环境与数据

In [ ]:
import importlib
import importlib.util
import os
import subprocess
import sys
import time

IN_COLAB = "google.colab" in sys.modules
if importlib.util.find_spec("mlarena") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mlarena-sdk"], check=True)

import numpy as np
import pandas as pd
import torch
import torchvision

torch.set_num_threads(3)          # 评测容器只有 3 核：按同样条件测量
print(f"torch {torch.__version__}, threads {torch.get_num_threads()}")

In [ ]:
def make_task(images_train, labels_train, images_test, labels_test, seed):
    """与挑战相同：置换像素位置和标签含义，并加入轻微噪声。"""
    rng = np.random.RandomState(seed)
    label_perm, pixel_perm = rng.permutation(10), rng.permutation(28 * 28)

    def permute(images):
        flat = images.reshape(len(images), -1)[:, pixel_perm].astype(np.float32) / 255.0
        flat += rng.normal(0, 0.015, flat.shape).astype(np.float32)
        flat = flat * rng.uniform(0.96, 1.04, (len(flat), 1)).astype(np.float32)
        flat += rng.uniform(-0.02, 0.02, (len(flat), 1)).astype(np.float32)
        return (np.clip(flat, 0, 1) * 255).astype(np.uint8).reshape(-1, 28, 28)

    return {"X_train": permute(images_train), "y_train": label_perm[labels_train].reshape(-1, 1).astype(np.int64),
            "X_test": permute(images_test), "y_test": label_perm[labels_test].astype(np.int64)}


mnist_train = torchvision.datasets.MNIST("data", train=True, download=True)
mnist_test = torchvision.datasets.MNIST("data", train=False, download=True)
task = make_task(mnist_train.data.numpy(), mnist_train.targets.numpy(),
                 mnist_test.data.numpy(), mnist_test.targets.numpy(), seed=1)
print({k: (v.shape, v.dtype) for k, v in task.items()})

## 2. Agent（提交的 `agent.py`）

In [ ]:
%%writefile agent.py
import math
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

TRAIN_BUDGET_S = 40.0       # 训练墙钟预算（60 s 期限，留足余量）
EPOCHS = 1000               # 上限很大：实际时长由 TRAIN_BUDGET_S 决定
BATCH = 256
LR = 0.2                    # SGD 峰值学习率
LR_MIN = 1e-5               # 余弦退火终点学习率
HIDDEN_LAYERS = [256, 256]  # 隐藏层宽度
LABEL_SMOOTH = 0.1          # 标签平滑
VAL_FRAC = 0.0              # 留出验证集比例（仅诊断用）；0 = 用全部数据
VERBOSE = False             # 是否打印每个 epoch 的记录


class Agent:
    def __init__(self, output_dim: int = 10, seed=None):
        torch.set_num_threads(3)
        self.output_dim = output_dim
        self.seed = 0 if seed is None else seed

    @staticmethod
    def _flatten(X):
        return torch.from_numpy(np.asarray(X).reshape(len(X), -1).astype(np.float32) / 255.0)

    def _build_model(self, in_dim):
        """MLP：Linear -> BN -> ReLU -> ... -> Linear"""
        layers, d = [], in_dim
        for h in HIDDEN_LAYERS:
            layers += [nn.Linear(d, h), nn.BatchNorm1d(h), nn.ReLU()]
            d = h
        layers.append(nn.Linear(d, self.output_dim))
        return nn.Sequential(*layers)

    def train(self, X_train, y_train):
        t0 = time.perf_counter()
        if VERBOSE:
            print(f"==== train() called on {len(X_train)} samples ====")
        torch.manual_seed(self.seed)
        X = self._flatten(X_train)
        self.mu, self.sd = X.mean(), X.std()               # 本次任务的统计量
        X = (X - self.mu) / self.sd
        y = torch.from_numpy(np.asarray(y_train).reshape(-1).astype(np.int64))

        X_val = y_val = None                               # 可选：诊断用验证集
        if VAL_FRAC > 0:
            perm = torch.randperm(len(X))
            n_val = int(len(X) * VAL_FRAC)
            X_val, y_val = X[perm[:n_val]], y[perm[:n_val]]
            X, y = X[perm[n_val:]], y[perm[n_val:]]
        self.history = []

        self.model = self._build_model(X.shape[1])         # 每次调用都新建模型
        opt = torch.optim.SGD(self.model.parameters(), lr=LR, momentum=0.9, nesterov=True)
        self._lr = LR
        for epoch in range(EPOCHS):
            self.model.train()
            t_ep = time.perf_counter()
            correct, seen = 0, 0
            for idx in torch.randperm(len(X)).split(BATCH):
                if len(idx) < 2:                           # BN 训练模式不能处理单样本 batch
                    continue
                # 按已用时间比例计算余弦学习率
                frac = min((time.perf_counter() - t0) / TRAIN_BUDGET_S, 1.0)
                self._lr = LR_MIN + (LR - LR_MIN) * 0.5 * (1.0 + math.cos(math.pi * frac))
                for g in opt.param_groups:
                    g["lr"] = self._lr

                out = self.model(X[idx])
                loss = F.cross_entropy(out, y[idx], label_smoothing=LABEL_SMOOTH)
                opt.zero_grad()
                loss.backward()
                opt.step()
                correct += (out.argmax(1) == y[idx]).sum().item()
                seen += len(idx)
                if time.perf_counter() - t0 > TRAIN_BUDGET_S:        # 墙钟时间保护
                    self._log(epoch, correct / seen, X_val, y_val, t_ep, t0)
                    return
            self._log(epoch, correct / seen, X_val, y_val, t_ep, t0)

    def _log(self, epoch, train_acc, X_val, y_val, t_ep, t0):
        """记录：训练准确率、验证准确率、学习率、本 epoch 耗时、累计耗时。"""
        val_acc = float("nan")
        if X_val is not None:
            self.model.eval()
            with torch.no_grad():
                val_acc = (self.model(X_val).argmax(1) == y_val).float().mean().item()
            self.model.train()
        rec = {"epoch": epoch + 1, "train_acc": round(train_acc, 4), "val_acc": round(val_acc, 4),
               "lr": f"{self._lr:.1e}",
               "epoch_s": round(time.perf_counter() - t_ep, 2),
               "elapsed_s": round(time.perf_counter() - t0, 1)}
        self.history.append(rec)
        if VERBOSE:
            print(rec)

    def predict(self, X_test):
        self.model.eval()
        with torch.no_grad():
            return self.model((self._flatten(X_test) - self.mu) / self.sd).argmax(1).tolist()

## 3. 本地评估

`evaluate_agent` 模拟挑战流程：计时 `train` / `predict`，计算准确率；准确率 ≥ 0.98 时运行 Fashion-MNIST 检查（需 ≥ 0.40）。

In [ ]:
import agent

if "RESULTS" not in globals():   # 重跑本单元格不会清空历史
    RESULTS = []


def evaluate_agent(name, deadline_s=60.0):
    importlib.reload(agent)                              # 读取最新的 agent.py
    bot = agent.Agent()
    t0 = time.perf_counter(); bot.train(task["X_train"], task["y_train"]); train_s = time.perf_counter() - t0
    t0 = time.perf_counter(); pred = np.asarray(bot.predict(task["X_test"])).ravel(); predict_s = time.perf_counter() - t0
    row = {"name": name, "accuracy": float((pred == task["y_test"]).mean()),
           "train_s": round(train_s, 1), "predict_s": round(predict_s, 1),
           "within_deadlines": train_s < deadline_s and predict_s < deadline_s}
    if row["accuracy"] >= 0.98:                          # 挑战的检查：同一个 agent，另一个数据集
        f_train = torchvision.datasets.FashionMNIST("data", train=True, download=True)
        f_test = torchvision.datasets.FashionMNIST("data", train=False, download=True)
        fashion = make_task(f_train.data.numpy(), f_train.targets.numpy(),
                            f_test.data.numpy(), f_test.targets.numpy(), seed=2)
        bot.train(fashion["X_train"], fashion["y_train"])
        row["fashion_accuracy"] = float((np.asarray(bot.predict(fashion["X_test"])).ravel() == fashion["y_test"]).mean())
    RESULTS.append(row)
    return pd.DataFrame(RESULTS)


evaluate_agent("final: SGD 0.2 + LS 0.1")

可选：用多个种子评估平均准确率和波动（不做 Fashion 检查，约 2 分钟）。

In [ ]:
MULTI = []

def report_multi(name, seeds=(0, 1, 2)):
    importlib.reload(agent)
    accs, times = [], []
    for s in seeds:
        bot = agent.Agent(seed=s)
        t0 = time.perf_counter(); bot.train(task["X_train"], task["y_train"]); times.append(time.perf_counter() - t0)
        pred = np.asarray(bot.predict(task["X_test"])).ravel()
        accs.append(float((pred == task["y_test"]).mean()))
    MULTI.append({"name": name, "mean": round(np.mean(accs), 4), "std": round(np.std(accs), 4),
                  "runs": [round(a, 4) for a in accs], "max_train_s": round(max(times), 1)})
    print(pd.DataFrame(MULTI).to_string(index=False))

# report_multi("final: SGD 0.2 + LS 0.1")

## 4. 提交

API key 从 Colab 的 *Secrets* 面板读取（名称 `MLARENA_API_KEY`），不要把 key 粘贴进单元格。

In [ ]:
import mlarena

if IN_COLAB:
    from google.colab import userdata
    MLARENA_API_KEY = userdata.get("MLARENA_API_KEY")
else:
    MLARENA_API_KEY = os.environ["MLARENA_API_KEY"]

client = mlarena.connect(api_key=MLARENA_API_KEY)
submission = client.submit(8, files=["agent.py"], submission_name="s17b-mlp256x2-bn-sgd0.2-ls0.1-cosine",
                           runtime={"language": "python", "framework": "torch"}, wait=True)
print(submission["status"]["status"], "--", submission["status"]["last_status_message"])